In [1]:
import os
import sys
sys.path.append(os.path.abspath('').rsplit('/', 1)[0] + '/code')

# import meslf.networks.pandapower_data as pp_data
import numpy as np
import pandas as pd
import pandapower

from meslf.networks.gas_network import GasNetwork, GasNode, GasLink, GasHalfLink
from meslf.networks.carrier import Gas
from meslf.networks.electrical_network import ElectricalNetwork, ElectricalNode, ElectricalLink, ElectricalHalfLink
from meslf.networks.heat_network import HeatNetwork, HeatNode, HeatLink, HeatHalfLink
from meslf.networks.carrier import Water
from meslf.networks.heterogeneous_network import HeterogeneousNetwork, HeterogeneousNode
from meslf.utils.constants import bar, mbar, kV, MW

In [2]:
def get_data(network_name):
    """
    Get data from pandapower based on the name of the network.

    Parameters
    ----------
        network_name : str
            Name of the network. Options:
                case4gs
                case5
                case6ww
                case9
                case14
                case24_ieee_rts
                case30
                case_ieee30
                case33bw
                case39
                case57
                case89pegase
                case118
                case145
                case_illinois200
                case300
                case1354pegase
                case1888rte
                case2848rte
                case2869pegase
                case3120sp
                case6470rte
                case6495rte
                case6515rte
                case9241pegase
                GBnetwork
                GBreducednetwork
                iceland

    Return
    ----------
        Return : pandapower.networks
            A pandapower network object
    """
    
    result = {}
    exec("data = pandapower.networks.{}()".format(network_name),
         globals(),
         result)
    return result["data"]

def find_node(name, network, node_names):
    return network.nodes[node_names.index(name)]

def compute_admittance(c_nf_per_km, f, g_us_per_km, length_km, parallel, r_ohm_per_km, x_ohm_per_km, Vn, Sn):
    Zn = Vn**2 / Sn
    r = r_ohm_per_km*length_km/parallel / Zn
    x = x_ohm_per_km*length_km/parallel / Zn
    
    b = -x / (r**2 + x**2)
    g = r / (r**2 + x**2)
    
    b_sh = 2*np.pi*f*c_nf_per_km*10**-9*length_km*parallel * Zn
    g_sh = g_us_per_km*10**-6*length_km*parallel * Zn

    return b, g, b_sh, g_sh

def compute_trafo(vn_hv_kv, vn_lv_kv, vn_kv_from, vn_kv_to, pfe_kw, i0_percent, vk_percent, vkr_percent, \
                  sn_mva, sn_mva_net, \
                  tap_neutral, tap_phase_shifter, tap_pos, tap_side, tap_step_degree, tap_step_percent, \
                  shift_degree):
    
    if (tap_phase_shifter == False) and (not np.isnan(tap_step_percent)) and (not np.isnan(tap_neutral)) and (not np.isnan(tap_pos)): # (tap_side in ['lv', 'hv']):
        n_tap = abs(1 + (tap_pos - tap_neutral) * tap_step_percent * 0.01)
        if tap_side == 'hv':
            vn_hv_kv *= n_tap
        elif tap_side =='lv':
            vn_lv_kv *= n_tap
            
    ratio = (vn_hv_kv / vn_lv_kv) * (vn_kv_to / vn_kv_from) 

    sn_ratio = sn_mva_net / sn_mva
    
    vn_ratio = (vn_kv_to / vn_lv_kv)**2
    
    Zn = vn_kv_to**2 / sn_mva_net
    Zn_trafo = vn_lv_kv**2 * sn_ratio

    vk_percent *= 0.01
    vkr_percent *= 0.01
    i0_percent *= 0.01

    r = vkr_percent * sn_ratio
    x = vk_percent * sn_ratio
    x = np.sign(vk_percent) * np.sqrt(x**2 - r**2)
    
    b = -x / (r**2 + x**2) * vn_ratio
    g = r / (r**2 + x**2) * vn_ratio

    g_sh = pfe_kw / (sn_mva * 1000) 
    b_sh = i0_percent**2 - g_sh**2
    if b_sh < 0:
        b_sh = 0
    b_sh = -np.sign(i0_percent) * np.sqrt(b_sh) * vn_ratio / sn_ratio
    g_sh *= vn_ratio / sn_ratio
             
    if False: # tap_phase_shifter:        
        if not np.isnan(tap_step_degree):
            shift_degree_tp = tap_step_degree*(tap_pos - tap_neutral)
        elif not np.isnan(tap_step_percent):
            shift_degree_tp = 2*np.arcsin(0.5*tap_step_percent*10**-2)*(tap_pos - tap_neutral)
        else:
            shift_degree_tp = 0
        
        phase_shift = (shift_degree + shift_degree_tp)*np.pi/180
    else:
        phase_shift = 0

    return b, g, b_sh, g_sh, ratio, phase_shift

def create_network_pandapower(network_name, ignore_nodes=[], no_slack=False):
    data = get_data(network_name)

    f = data.f_hz # default frequency

    network = ElectricalNetwork(network_name)
    
    node_name = ''
    node_names = []

    data.gen.loc[data.gen["sn_mva"].isnull(), 'sn_mva'] = data.sn_mva
    data.load.loc[data.load["sn_mva"].isnull(), 'sn_mva'] = data.sn_mva
    data.sgen.loc[data.sgen["sn_mva"].isnull(), 'sn_mva'] = data.sn_mva

    # slack node
    if no_slack:
        first = True
        for index, row in data.ext_grid.iterrows():
            if row["in_service"]:
                node_name = data.bus.loc[row['bus'], 'name']
                if node_name not in ignore_nodes:
                    if first:
                        network.add_node(ElectricalNode(name=node_name,
                                                        P=0,
                                                        V=row["vm_pu"],
                                                        delta=row["va_degree"]*np.pi/180,
                                                        node_type=3,
                                                        Sbase=data.sn_mva, # slack has no base complex power, so I set default to default of network
                                                        Vbase=data.bus.loc[row['bus'], 'vn_kv']))
                        first = False
                    else:
                        network.add_node(ElectricalNode(name=node_name,
                                                        V=row["vm_pu"],
                                                        delta=row["va_degree"]*np.pi/180,
                                                        node_type=0,
                                                        Sbase=data.sn_mva, # slack has no base complex power, so I set default to default of network
                                                        Vbase=data.bus.loc[row['bus'], 'vn_kv']))
                    node_names.append(node_name)
    else:
        for index, row in data.ext_grid.iterrows():
            if row["in_service"]:
                node_name = data.bus.loc[row['bus'], 'name']
                if node_name not in ignore_nodes:
                    network.add_node(ElectricalNode(name=node_name,
                                                    V=row["vm_pu"],
                                                    delta=row["va_degree"]*np.pi/180,
                                                    node_type=0,
                                                    Sbase=data.sn_mva, # slack has no base complex power, so I set default to default of network
                                                    Vbase=data.bus.loc[row['bus'], 'vn_kv']))
                    node_names.append(node_name)
                else:
                    print("Slack node is not included in network. This leads to an ill-posed system.")

    # PV node (P=-P, V=V)
    for index, row in data.gen.iterrows():
        if row["in_service"]:
            node_name = data.bus.loc[row['bus'], 'name']
            if node_name not in node_names + ignore_nodes:
                network.add_node(ElectricalNode(name=node_name,
                                                P=-row["p_mw"]/row["sn_mva"],
                                                V=row["vm_pu"],
                                                node_type=1,
                                                Sbase=row['sn_mva'],
                                                Vbase=data.bus.loc[row['bus'], 'vn_kv']))      
                node_names.append(data.bus.loc[row['bus'], 'name'])
            elif node_name in node_names:
                ElectricalHalfLink('{}_pv_{}_{}'.format(node_name, -row["p_mw"], row["vm_pu"]), 
                                find_node(node_name, network, node_names),
                                bc_type=2,
                                P=-row["p_mw"]/row["sn_mva"])

    # PQ node (static generator, P=-P, Q=-Q)
    for index, row in data.sgen.iterrows():
        if row["in_service"]:
            node_name = data.bus.loc[row['bus'], 'name']
            if node_name not in node_names + ignore_nodes:
                network.add_node(ElectricalNode(name=node_name,
                                                P=-row["p_mw"]/row["sn_mva"],
                                                Q=-row["q_mvar"]/row["sn_mva"],
                                                node_type=2,
                                                Sbase=row['sn_mva'],
                                                Vbase=data.bus.loc[row['bus'], 'vn_kv']))      
                node_names.append(data.bus.loc[row['bus'], 'name'])
            elif node_name in node_names:
                ElectricalHalfLink('{}_pq_{}_{}'.format(node_name, -row["p_mw"], -row["q_mvar"]), 
                                   find_node(node_name, network, node_names),
                                   bc_type=1,
                                   P=-row["p_mw"]/row["sn_mva"],
                                   Q=-row["q_mvar"]/row["sn_mva"])

    # Load (PQ node)
    for index, row in data.load.iterrows():
        if row["in_service"]:
            node_name = data.bus.loc[row['bus'], 'name']
            if node_name not in node_names + ignore_nodes:
                network.add_node(ElectricalNode(name=node_name,
                                                P=row["p_mw"]/row["sn_mva"],
                                                Q=row["q_mvar"]/row["sn_mva"],
                                                node_type=2,
                                                Sbase=row["sn_mva"],
                                                Vbase=data.bus.loc[row['bus'], 'vn_kv']))
                node_names.append(node_name)
            elif node_name in node_names:
                ElectricalHalfLink('{}_pq_{}_{}'.format(node_name, row["p_mw"], row["q_mvar"]), 
                                   find_node(node_name, network, node_names),
                                   bc_type=1,
                                   P=row["p_mw"]/row["sn_mva"], 
                                   Q=row["q_mvar"]/row["sn_mva"])
                        
    # Junction (PQ Node, P=0, Q=0)
    for index, row in data.bus.iterrows():
        if row['name'] not in node_names + ignore_nodes:
            network.add_node(ElectricalNode(name=row['name'],
                                            P=None,
                                            Q=None,
                                            node_type=2,
                                            Sbase=data.sn_mva,
                                            Vbase=row["vn_kv"]))
            node_names.append(row['name'])

    # Shunt (PQ Node, P=P, Q=-Q)
    for index, row in data.shunt.iterrows():
        if row["in_service"]:
            node_name = data.bus.loc[row['bus'], 'name']
            if node_name in node_names:
                P = row["p_mw"]*row["step"] / data.sn_mva
                Q = -row["q_mvar"]*row["step"] / data.sn_mva # injected reactive power
                y = (P + 1j*Q)
                ElectricalHalfLink('{}_shunt_{}_{}'.format(node_name, P, Q), 
                                   find_node(node_name, network, node_names),
                                   bc_type=1,
                                   P=P, 
                                   Q=Q,
                                   link_type='nodal_shunt',
                                   link_params={'b' : y.imag,
                                                'g' : y.real})

    # Line
    for index, row in data.line.iterrows():
        if row["in_service"]:
            from_bus = data.bus.loc[row["from_bus"], 'name']
            to_bus = data.bus.loc[row["to_bus"], 'name']
            if row["from_bus"] not in ignore_nodes and row["to_bus"] not in ignore_nodes:
                b, g, b_sh, g_sh = compute_admittance(c_nf_per_km=row["c_nf_per_km"],
                                                      f=f,
                                                      g_us_per_km=row["g_us_per_km"],
                                                      length_km=row["length_km"],
                                                      parallel=row["parallel"],
                                                      r_ohm_per_km=row["r_ohm_per_km"],
                                                      x_ohm_per_km=row["x_ohm_per_km"],
                                                      Vn=data.bus.loc[row["from_bus"]].vn_kv,
                                                      Sn=find_node(from_bus, network, node_names).Sbase)
                
                network.add_link(ElectricalLink(name="{}-{}".format(from_bus, to_bus),
                                                start_node=find_node(from_bus, network, node_names),
                                                end_node=find_node(to_bus, network, node_names),
                                                bc_type=0,
                                                link_type='pi_line',
                                                link_params={'b' : b,
                                                             'g' : g,
                                                             'b_sh' : b_sh,
                                                             'g_sh' : g_sh}))
            
    # Transformer
    for index, row in data.trafo.iterrows():
        if row["in_service"]:
            from_bus = data.bus.loc[row["hv_bus"], 'name']
            to_bus = data.bus.loc[row["lv_bus"], 'name']
            if row["hv_bus"] not in ignore_nodes and row["lv_bus"] not in ignore_nodes:
                b, g, b_sh, g_sh, ratio, phase_shift = compute_trafo(vn_hv_kv=row["vn_hv_kv"],
                                                                     vn_lv_kv=row["vn_lv_kv"],
                                                                     vn_kv_from=data.bus.loc[row["hv_bus"]].vn_kv,
                                                                     vn_kv_to=data.bus.loc[row["lv_bus"]].vn_kv,
                                                                     pfe_kw=row["pfe_kw"],
                                                                     i0_percent=row["i0_percent"],
                                                                     vk_percent=row["vk_percent"],
                                                                     vkr_percent=row["vkr_percent"],
                                                                     sn_mva=row["sn_mva"],
                                                                     sn_mva_net=data.sn_mva,
                                                                     tap_neutral=row["tap_neutral"],
                                                                     tap_phase_shifter=row["tap_phase_shifter"],
                                                                     tap_pos=row["tap_pos"],
                                                                     tap_side=row["tap_side"],
                                                                     tap_step_degree=row["tap_step_degree"],
                                                                     tap_step_percent=row["tap_step_percent"],
                                                                     shift_degree=row["shift_degree"])
                
                # print("{}-{} : g_sh = {}, b_sh = {}, ratio = {}".format(from_bus, to_bus, g_sh, b_sh, ratio))
                
                network.add_link(ElectricalLink(name="{}-{}".format(from_bus, to_bus),
                                                start_node=find_node(from_bus, network, node_names),
                                                end_node=find_node(to_bus, network, node_names),
                                                bc_type=0,
                                                link_type='pi_line_trafo',
                                                link_params={'b' : b,
                                                             'g' : g,
                                                             'b_sh' : b_sh,
                                                             'g_sh' : g_sh,
                                                             'ratio' : ratio,
                                                             'phase_shift' : phase_shift}))
                
    return network, data

# def find_index(data, name):
#     return pd.Index(data).get_loc(name)

# def create_directory_for_results():
#     network_names = [
#                      "case4gs",
#                      "case5",
#                      "case6ww",
#                      "case9",
#                      "case14",
#                      "case24_ieee_rts",
#                      "GBreducednetwork",
#                      "case30",
#                      "case_ieee30",
#                      "case33bw",
#                      "case39",
#                      "case57",
#                      "case89pegase",
#                      "case118",
#                      "case145",
#                      "iceland",
#                      "case_illinois200",
#                      "case300",
#                      "case1354pegase",
#                      "case1888rte",
#                      "GBnetwork",
#                      "case2848rte",
#                      "case2869pegase",
#                      "case3120sp",
#                      "case6470rte",
#                      "case6495rte",
#                      "case6515rte",
#                      "case9241pegase"]

#     dir_path = os.path.join(os.path.abspath(''), 'results')

#     if not os.path.exists(dir_path):
#         os.makedirs(dir_path)

#     for network_name in network_names:
#         sub_dir_path = os.path.join(dir_path, network_name)
#         if not os.path.exists(sub_dir_path):
#             os.makedirs(sub_dir_path)
        
#         for method in ['direct', 'gmres', 'bicgstab', 'idr', 'lsqr']:
#             path = os.path.join(sub_dir_path, method)
#             if not os.path.exists(path):
#                 os.makedirs(path)

In [3]:
# %% gas

S = 0.0696  # 0.589  # This value to ensure equivalence between different pipe models
Z = 1.
pn = 1.*bar  # [Pa]
Tn = 288.  # [K]
T = Tn
R = 8.314413  # [J/molK]
M = 28.97e-3  # [kg/mol]
R_air = R/M  # [J/kgK]
gas = Gas('hydrogen gas', S, R_air, Z, pn, Tn, T)
rhon_g = gas.rhon  # [kg/m^3]

# physical parameters of network and pipes
L_g = 1000.  # [m]
D_g = 1  # [m]
gas_link_type = 'pipe_high_pres_pole'
gas_link_params = {'L': L_g, 'D': D_g, 'carrier': gas}
link_eq = 'dp_of_q'

# physical parameters of the coupling unit
eta = 8/10
eta_ratio = 1/6
LHV = 33.33*3600*10**3 / MW # [MJ/kg]
HHV = 1.418*10**8 / MW # 60134305 # [MJ/kg]
unit_type = 'p2g'
unit_params = {'eta': eta,
               'eta_ratio': eta_ratio,
               'GHV': LHV}

network_names = [
                 "case4gs",
                 "case5",
                 "case6ww",
                 "case9",
                 "case14",
                 "case24_ieee_rts",
                 "GBreducednetwork",
                 "case30",
                 "case_ieee30",
                 "case33bw",
                 "case39",
                 "case57",
                 "case89pegase",
                 "case118",
                 "case145",
                 "iceland",
                 "case_illinois200",
                 "case300",
                 "case1354pegase",
                 "case1888rte",
                 "GBnetwork",
                 "case2848rte",
                 "case2869pegase",
                 "case3120sp",
                 "case6470rte",
                 "case6495rte",
                 "case6515rte",
                 "case9241pegase"]

<h1> P2G
<h3> Supply - 1 node
<h5> Gas reference node <br>
Electrical PVdelta node

In [4]:
first = True

for network_name in network_names[:2]:
    # create electrical network
    elec_net, data_pp = create_network_pandapower(network_name=network_name, no_slack=True)

    # create gas network
    p_ref = 80 # bar
    
    gas_net = GasNetwork('Gas')
    
    g0 = GasNode('g0', node_type=0, p=p_ref*bar) # p
    gas_net.add_node(g0)
    # GasHalfLink('g0_hl', g0, bc_type=1, q=1)
        
    # create integrated network
    het_net = HeterogeneousNetwork(network_name + ' + ' + 'gas_1_node')

    # add single-carriers to integrated network
    het_net.add_network(gas_net)
    het_net.add_network(elec_net)

    # create coupling unit
    unit_params = {'eta': eta,
                   'eta_ratio': eta_ratio,
                   'GHV': LHV / data_pp.sn_mva}
    
    c0 = HeterogeneousNode('p2g', x=0, y=0,
                           node_type=1,
                           unit_type='p2g',
                           unit_params=unit_params)
    het_net.add_node(c0)

    # dummy links
    c0g0 = GasLink('c0g0', c0, g0)
    gas_net.add_link(c0g0)

    e0c0 = ElectricalLink('e0c0', elec_net.nodes[0], c0,
                          bc_type=2,
                          Qstart=0)
    elec_net.add_link(e0c0)

    formulation = {'gas': 'full',
                   'elec': 'complex_power',
                   'het': None}

    scale_var = "per_unit"

    scale_var_params = {'qbase': 1,
                        'pbase': p_ref*bar,
                        'Sbase': 1,
                        'Vbase': 1,
                        'deltabase': 1,
                        'Ebase' : 1}

    tol = 10**-6
    max_iter = 20

    q_init = np.ones(len([l for l in gas_net.links if l.link_type != "dummy"])) # flat start 1
    p_init = p_ref*bar*np.linspace(1.05, 1.01, len(gas_net.nodes)-1) # linear profile
    delta_init = np.zeros(len(het_net.get_x_entries(formulation=formulation)[3])) # flat start 0
    V_init = np.ones(len(het_net.get_x_entries(formulation=formulation)[4])) # flat start 1
    qc_init = np.ones(len(het_net.get_x_entries(formulation=formulation)[14])) # flat start 1
    Pc_init = np.ones(len(het_net.get_x_entries(formulation=formulation)[16])) # flat start 1

    het_net.initialize()
    het_net.update(np.concatenate([q_init, p_init, delta_init, V_init, qc_init, Pc_init]), formulation=formulation)

    x_sol, iterations, errors, p_g_vec, q_vec, q_inj, \
    delta_vec, V_mag_vec, S_inj, P_edge, Q_edge, m_vec, \
    p_h_vec, Ts_vec, Tr_vec, m_hl_vec, phi_hl_vec, Ts_hl_vec, \
    Tr_hl_vec, qc_vec, Pc_vec, Qc_vec, mc_vec, phic_vec, \
    Tsc_vec, Trc_vec= het_net.solve_network(tol,
                                            max_iter,
                                            solver='NR',
                                            formulation=formulation,
                                            scale_var=scale_var,
                                            scale_var_params=scale_var_params)
    
    interim_data = "{:<18s} & {:<6s} & {:3d} & {:>.20f} \\\\ \\hline\n".format(network_name, str(errors[-1] < tol), iterations, errors[-1])
    print(interim_data, end="")
    
    # path = os.path.abspath('')
    # save_path = os.path.join(path, 'results', 'pandapower', 'demand', '1node')
    # os.makedirs(save_path, exist_ok=True)
    # if first:
    #     open(os.path.join(save_path, 'residual_nr.txt'), 'w').close()
    # f = open(os.path.join(save_path, 'residual_nr.txt'), 'a')
    # f.write(interim_data)
    # f.close()
    
    # path = os.path.abspath('')
    # save_path = os.path.join(path, 'results', 'pandapower', 'demand', '1node', '{}'.format(network_name))
    # os.makedirs(save_path, exist_ok=True)
    
    # if first:
    #     open(os.path.join(save_path, 'gas_pressure.txt'), 'w').close()
    #     open(os.path.join(save_path, 'gas_injected_flow.txt'), 'w').close()
    #     open(os.path.join(save_path, 'gas_flow.txt'), 'w').close()
    #     open(os.path.join(save_path, 'delta.txt'), 'w').close()
    #     open(os.path.join(save_path, 'voltage.txt'), 'w').close()
    #     open(os.path.join(save_path, 'injected_complex_power.txt'), 'w').close()
    #     open(os.path.join(save_path, 'active_power.txt'), 'w').close()
    #     open(os.path.join(save_path, 'reactive_power.txt'), 'w').close()
    #     open(os.path.join(save_path, 'coupling_gas_flow.txt'), 'w').close()
    #     open(os.path.join(save_path, 'coupling_active_power.txt'), 'w').close()
    #     open(os.path.join(save_path, 'coupling_reactive_power.txt'), 'w').close()        
        
    #     first = False
    
    # np.savetxt(os.path.join(save_path, 'residual_nr.txt'), errors)      
    # np.savetxt(os.path.join(save_path, 'gas_pressure.txt'), p_g_vec)
    # np.savetxt(os.path.join(save_path, 'gas_injected_flow.txt'), q_inj)
    # np.savetxt(os.path.join(save_path, 'gas_flow.txt'), q_vec)
    # np.savetxt(os.path.join(save_path, 'delta.txt'), delta_vec)
    # np.savetxt(os.path.join(save_path, 'voltage.txt'), V_mag_vec)
    # np.savetxt(os.path.join(save_path, 'injected_complex_power.txt'), S_inj)
    # np.savetxt(os.path.join(save_path, 'active_power.txt'), P_edge)
    # np.savetxt(os.path.join(save_path, 'reactive_power.txt'), Q_edge)
    # np.savetxt(os.path.join(save_path, 'coupling_gas_flow.txt'), qc_vec)
    # np.savetxt(os.path.join(save_path, 'coupling_active_power.txt'), Pc_vec)
    # np.savetxt(os.path.join(save_path, 'coupling_reactive_power.txt'), Qc_vec)

case4gs            & True   &   3 & 0.00000000145189507080 \\ \hline
case5              & True   &   3 & 0.00000000007123397977 \\ \hline
